# Tracking + play-by-play exploration

Look at raw rows from a tracking Parquet and the play-by-play Parquet for the same game, side by side. Exploration only; nothing here is a pipeline dependency (see PROJECT.md).

Run this with the project's `.venv` as the kernel so `nba_assists` is already importable (`.venv/bin/pip install -e ".[dev]"`).

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

from nba_assists.config import get_config

cfg = get_config()
tracking_dir = cfg.get_path("tracking")
pbp_dir = cfg.get_path("pbp")
pd.set_option("display.max_columns", None)

## Pick a game

Any `game_id` that has both tracking and play-by-play works. Change `GAME_ID` to look at a different one.

In [ ]:
con = duckdb.connect()
GAME_ID = con.execute(
    f"SELECT DISTINCT game_id FROM read_parquet('{tracking_dir}/*.parquet') ORDER BY game_id LIMIT 1"
).fetchone()[0]

tracking_path = Path(
    con.execute(
        f"SELECT filename FROM read_parquet('{tracking_dir}/*.parquet', filename=true) "
        "WHERE game_id = ? LIMIT 1",
        [GAME_ID],
    ).fetchone()[0]
)
con.close()

print("game_id:", GAME_ID)
print("tracking file:", tracking_path)

## Tracking rows

One row per tracked object per frame (`ts_ms`). A full frame has 10 player rows plus 1 ball row (`team_id`/`player_id` equal to the ball sentinels in `config.yaml`).

In [ ]:
tracking = pd.read_parquet(tracking_path)
print(tracking.shape)
tracking.head(15)

In [ ]:
# every row sharing the first frame's ts_ms -- should be 10 players + 1 ball
first_ts = tracking["ts_ms"].iloc[0]
tracking[tracking["ts_ms"] == first_ts]

## Play-by-play rows, same game

Already sorted by `PERIOD`, then `clock_s` descending (see `scripts/prepare_pbp.py`).

In [ ]:
pbp = pd.read_parquet(pbp_dir / "pbp_2015_16.parquet")
game_pbp = pbp[pbp["GAME_ID"] == GAME_ID].reset_index(drop=True)
print(game_pbp.shape)
game_pbp.head(15)

## Cross-referencing one event

Take the game's first made shot from play-by-play, then look at the tracking ball frames around its recorded `clock_s` -- the same kind of window `scripts/detect_makes.py` searches (see `rim_detection:` in `config.yaml`).

In [ ]:
made_shot_type = int(cfg.pbp.event_types.made_shot)
make_row = game_pbp[game_pbp["EVENTMSGTYPE"] == made_shot_type].iloc[0]
make_row[["EVENTNUM", "PERIOD", "clock_s", "HOMEDESCRIPTION", "VISITORDESCRIPTION"]]

In [ ]:
period = int(make_row["PERIOD"])
clock_s = float(make_row["clock_s"])
ball_team_id = int(cfg.sentinels.ball_team_id)
ball_player_id = int(cfg.sentinels.ball_player_id)

nearby_ball_frames = tracking[
    (tracking["quarter"] == period)
    & (tracking["team_id"] == ball_team_id)
    & (tracking["player_id"] == ball_player_id)
    & tracking["game_clock"].between(clock_s - 2, clock_s + 2)
].sort_values("ts_ms")[["ts_ms", "game_clock", "x", "y", "z"]]
nearby_ball_frames